In [1]:
import numpy as np
import cv2

# Constants
WINDOW_NAME = "Draw Rectangle"
BACKGROUND_COLOR = (255, 255, 255)
RECTANGLE_COLOR = (0, 255, 0)
CANVAS_SIZE = (1080, 1920, 3)

# Global Variables
background = np.ones(CANVAS_SIZE, dtype=np.uint8) * 255
image = background.copy()
current_frame = image.copy()
drawing = False
start_point = None
end_point = None
rectangle = None


# Helper Functions
def reset_canvas():
    global image, current_frame
    image = background.copy()
    current_frame = image.copy()


def draw_preview_rectangle(x, y):
    global current_frame
    temp_image = image.copy()
    cv2.rectangle(temp_image, start_point, (x, y), RECTANGLE_COLOR, 2)
    current_frame = temp_image


def draw_permanent_rectangle():
    global image, current_frame, rectangle
    cv2.rectangle(image, start_point, end_point, RECTANGLE_COLOR, 2)
    current_frame = image.copy()
    rectangle = (start_point, end_point)
    print(f"Rectangle coordinates: {start_point} to {end_point}")


def apply_transformation(corners, transformation_matrix):
    transformed_corners = np.zeros_like(corners)
    for i in range(corners.shape[0]):
        transformed_corners[i] = transformation_matrix @ corners[i, :]
    return transformed_corners


def update_rectangle(corners):
    global start_point, end_point, rectangle
    start_point = (int(corners[0][0]), int(corners[0][1]))
    end_point = (int(corners[3][0]), int(corners[3][1]))
    rectangle = (start_point, end_point)
    reset_canvas()
    draw_permanent_rectangle()


def get_corners():
    global start_point, end_point
    return np.array(
        [
            [start_point[0], start_point[1], 1],
            [end_point[0], start_point[1], 1],
            [start_point[0], end_point[1], 1],
            [end_point[0], end_point[1], 1],
        ]
    )


def get_centered_transformation(transformation_matrix):
    global start_point, end_point

    # Compute the center of the rectangle
    rectangle_center = (
        (start_point[0] + end_point[0]) / 2,
        (start_point[1] + end_point[1]) / 2,
    )

    # Move to origin
    T_to_origin = np.array(
        [[1, 0, -rectangle_center[0]], [0, 1, -rectangle_center[1]], [0, 0, 1]]
    )

    # Move back
    T_back = np.array(
        [[1, 0, rectangle_center[0]], [0, 1, rectangle_center[1]], [0, 0, 1]]
    )

    # Combine transformation
    T = T_back @ transformation_matrix @ T_to_origin
    return T


# Transformation Functions
def translate_rectangle():
    global rectangle, start_point, end_point
    if rectangle is None:
        print("No rectangle to translate")
        return

    tx = int(input("Enter the translation in the x-direction: "))
    ty = int(input("Enter the translation in the y-direction: "))
    T = np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]])
    corners = get_corners()
    translated_corners = apply_transformation(corners, T)
    update_rectangle(translated_corners)


def rotate_rectangle():
    global rectangle, start_point, end_point
    if rectangle is None:
        print("No rectangle to rotate")
        return

    theta = np.radians(int(input("Enter the rotation angle in degrees: ")))
    R = np.array(
        [
            [np.cos(theta), -np.sin(theta), 0],
            [np.sin(theta), np.cos(theta), 0],
            [0, 0, 1],
        ]
    )
    T = get_centered_transformation(R)
    corners = get_corners()
    rotated_corners = apply_transformation(corners, T)
    update_rectangle(rotated_corners)


def scale_rectangle():
    global rectangle, start_point, end_point
    if rectangle is None:
        print("No rectangle to scale")
        return

    sx = float(input("Enter the scaling factor in the x-direction: "))
    sy = float(input("Enter the scaling factor in the y-direction: "))
    S = np.array([[sx, 0, 0], [0, sy, 0], [0, 0, 1]])
    T = get_centered_transformation(S)
    corners = get_corners()
    scaled_corners = apply_transformation(corners, T)
    update_rectangle(scaled_corners)


# Mouse Event Handling
def mouse_event(event, x, y, flags, param):
    global drawing, start_point, end_point
    if event == cv2.EVENT_LBUTTONDOWN:
        reset_canvas()
        drawing = True
        start_point = (x, y)
    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            draw_preview_rectangle(x, y)
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        end_point = (x, y)
        draw_permanent_rectangle()


# Main Loop
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.moveWindow(WINDOW_NAME, 40, 30)
cv2.setMouseCallback(WINDOW_NAME, mouse_event)

while True:
    cv2.imshow(WINDOW_NAME, current_frame)
    key = cv2.waitKey(1) & 0xFF  # Keeping only the last 8 bits of the key code

    if key == ord("t"):  # Press 't' to translate
        translate_rectangle()
    elif key == ord("r"):  # Press 'r' to rotate
        rotate_rectangle()
    elif key == ord("s"):  # Press 's' to scale
        scale_rectangle()
    elif key == 27:  # ESC key
        break

cv2.destroyAllWindows()

Rectangle coordinates: (430, 159) to (1135, 558)
Rectangle coordinates: (982, 6) to (583, 711)
Rectangle coordinates: (1135, 558) to (430, 159)
Rectangle coordinates: (583, 711) to (982, 6)
Rectangle coordinates: (430, 159) to (1135, 558)
Rectangle coordinates: (430, -40) to (1135, 757)
Rectangle coordinates: (430, 160) to (1135, 957)
